In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Load your dataset
df= pd.read_csv(r'C:\Users\user\Smart_Waste_collection_Route_Optimization\Data\amravati_historical_waste.csv')

# 2. Strategic Feature Engineering
# Capturing the Mid-week Peak you identified in EDA
df['is_wednesday_peak'] = df['days'].apply(lambda x: 1 if x == 'Wednesday' else 0)

# Capturing the Sunday Peak
df['is_sunday_peak'] = df['days'].apply(lambda x: 1 if x == 'Sunday' else 0)

# Flagging Depot points so the model handles 40+ hour outliers separately
df['is_depot'] = df['Area_Type'].apply(lambda x: 1 if x == 'Depot' else 0)

# 3. Encoding Categorical Data
# Converting 'Area_Type' into numbers (Market=0, Depot=1, etc.)
le_area = LabelEncoder()
df['Area_Type_Encoded'] = le_area.fit_transform(df['Area_Type'])

# 4. Feature Scaling (The 0 to 1 Normalization)
# We scale current fill and fill rate so the model doesn't favor large numbers
scaler = StandardScaler()
numerical_features = ['Current_Fill_Pct', 'Actual_Fill_Rate']
df[numerical_features] = scaler.fit_transform(df[numerical_features])

# 5. Selecting Final Features for Modeling
# We exclude Lat/Lon for now (those are for the Routing phase, not the Urgency phase)
X = df[['Area_Type_Encoded', 'Current_Fill_Pct', 'Actual_Fill_Rate', 
        'is_wednesday_peak', 'is_sunday_peak', 'is_depot']]

# 6. Defining the Target variable (Example: Hours_Left)
# Our goal is to predict how much time is left before overflow
y = df['Hours_Left']

print("Stage 1 Complete: Data is now normalized and stratified.")
print(X.head())

Stage 1 Complete: Data is now normalized and stratified.
   Area_Type_Encoded  Current_Fill_Pct  Actual_Fill_Rate  is_wednesday_peak  \
0                  3         -1.081829          1.465683                  1   
1                  5         -1.532215          0.289648                  0   
2                  2         -0.901674         -1.236433                  0   
3                  0          0.224293          0.483095                  0   
4                  0         -1.532215          0.492307                  0   

   is_sunday_peak  is_depot  
0               0         0  
1               0         0  
2               0         0  
3               0         0  
4               0         0  


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd

# 1. Split data into Training and Testing sets
# X contains our features from Stage 1, y is the 'Hours_Left'
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Initialize the Random Forest Regressor
# n_estimators=100 means we are using 100 decision trees in our forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# 3. Train the model
rf_model.fit(X_train, y_train)

# 4. Make predictions
y_pred = rf_model.predict(X_test)

# 5. Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("Model Evaluation Complete.")

print(f"Model Performance:")
print(f"Mean Absolute Error: {mae:.2f} hours")
print(f"R-squared Score: {r2:.2f}")

# 6. Feature Importance - This is the 'Winning' insight for your hackathon!
importances = rf_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

print("\nWhat the AI thinks is most important:")
print(feature_importance_df)

Model Evaluation Complete.
Model Performance:
Mean Absolute Error: 1.39 hours
R-squared Score: 0.90

What the AI thinks is most important:
             Feature  Importance
2   Actual_Fill_Rate    0.681888
1   Current_Fill_Pct    0.266525
3  is_wednesday_peak    0.028776
5           is_depot    0.018801
0  Area_Type_Encoded    0.002931
4     is_sunday_peak    0.001079


In [4]:
y_pred_of_train=rf_model.predict(X_train)
mae1 = mean_absolute_error(y_train, y_pred_of_train)
print(f"Mean Absolute Error: {mae1:.2f} hours")

Mean Absolute Error: 0.68 hours
